# ME 280 HW 1

I am starting to like these workbooks. Though, I would still like it to be 100% markdown with runnable block-code cells. Anyway, before I do anything, I wanted to create an enum for all cards. According to [the official docs](https://docs.python.org/3/library/enum.html), there's no enum keyword in this language but there is an abstract class to extend from.


In [10]:
from enum import Enum


# Enums are just class inherence here, nothing special here.
# I personally like CamelCase for the entries but it's acceptable
# in my eyes to just follow the docs and do SCREAMING_SNAKE_CASE.
# Oh also, the numbers here are arbitrary, they're not the actual values.
class Card(Enum):

    # I gave them discriminators somewhat matching with their values
    # so that makes it a 1-based indexing solution. I will handle the
    # worth of aces and face cards in a separate method.
    ACE = 1
    TWO = 2
    THREE = 3
    FOUR = 4
    FIVE = 5
    SIX = 6
    SEVEN = 7
    EIGHT = 8
    NINE = 9
    TEN = 10
    JACK = 11
    QUEEN = 12
    KING = 13

    # Declaring a worth method here since the indices are arbitrary. Furthermore,
    # I have also added the treat_ace_as_1 parameter to toggle between the
    # two possible ways of treating an ace as 1 or 11. My most used language is
    # TypeScript so I will not be hesitating from using type annotations.
    def worth(self, treat_ace_as_1=True) -> int:

        # Handling the ace is simple.
        if self == Card.ACE:

            # I am not sure if I like Python's ternary operator syntax but
            # I guess I am glad it even exists in the first place.
            return 1 if treat_ace_as_1 else 11

        # This solution to checking if the card falls in this list of face cards
        # is clean but I don't seem to be able to move the list into a static
        # field. This is definitely a performance hit.
        elif self in [Card.JACK, Card.QUEEN, Card.KING]:
            return 10

        # The rest of the cards have the correct values in their
        # enum discriminators.
        else:
            return self.value

    # During debugging, I found it useful to use the card emojis to
    # view the deck. It remains unused for the final submission.
    def __str__(self):

        # Switch cases for the win! Emojis from
        # https://www.piliapp.com/emoji/list/playing-cards/
        case = {
            Card.ACE: "🂱",
            Card.TWO: "🂲",
            Card.THREE: "🂳",
            Card.FOUR: "🂴",
            Card.FIVE: "🂵",
            Card.SIX: "🂶",
            Card.SEVEN: "🂷",
            Card.EIGHT: "🂸",
            Card.NINE: "🂹",
            Card.TEN: "🂺",
            Card.JACK: "🂻",
            Card.QUEEN: "🂽",
            Card.KING: "🂾",
        }

        return case[self]

It would be a great idea to test all these methods and variants out.


In [11]:
print(Card.ACE, Card.ACE.worth())
print(Card.ACE, Card.ACE.worth(False))
print(Card.TWO, Card.TWO.worth())
print(Card.THREE, Card.THREE.worth())
print(Card.FOUR, Card.FOUR.worth())
print(Card.FIVE, Card.FIVE.worth())
print(Card.SIX, Card.SIX.worth())
print(Card.SEVEN, Card.SEVEN.worth())
print(Card.EIGHT, Card.EIGHT.worth())
print(Card.NINE, Card.NINE.worth())
print(Card.TEN, Card.TEN.worth())
print(Card.JACK, Card.JACK.worth())
print(Card.QUEEN, Card.QUEEN.worth())
print(Card.KING, Card.KING.worth())

🂱 1
🂱 11
🂲 2
🂳 3
🂴 4
🂵 5
🂶 6
🂷 7
🂸 8
🂹 9
🂺 10
🂻 10
🂽 10
🂾 10


Another every important thing I would like to implement is all the different sorts of player behavior my "game engine" can support. Initially I started off with just the dealer and the player but I realized there are a lot of funny names and behaviors I can come up with. Here's the 4 that I have at the moment:

- Drunk Gambler: this guy does a coin flip on either to hit or stand.
- High School Statistician: this person actually paid attention to statistics in high school and aims to stand just before `21 - average - 1` so that it's unlikely to bust if he were to go again. The `average` here is the average worth of all available cards.
- Seasoned Rich Guy: this guy looked up how to play Black Jack once on Google and has been using the same strategy ever since. Doesn't matter if he wins or now, he's rich so he doesn't care and won't change.
- Dealer: the dealer follows the standard <17 hit rule. Because of how I have this set up, you can make the dealer play against another dealer if you really wanted to. Actually, this is exactly what I do in the CSV file.


In [12]:
class PlayerBehavior(Enum):

    # Naming enums became real fun here.
    DRUNK_GAMBLER = 0
    HIGH_SCHOOL_STATISTICIAN = 1
    SEASONED_RICH_GUY = 2
    DEALER = 3

    # String representation will be useful when generating the CSV. The
    # SCREAMING_SNAKE_CASE is nice for coding but awful for sheets.
    def __str__(self):
        case = {
            PlayerBehavior.DRUNK_GAMBLER: "Drunk Gambler",
            PlayerBehavior.HIGH_SCHOOL_STATISTICIAN: "High School Statistician",
            PlayerBehavior.SEASONED_RICH_GUY: "Seasoned Rich Guy",
            PlayerBehavior.DEALER: "Dealer",
        }

        return case[self]

Before I can create the game flow logic, it'd be smart to model the players. In fact, I intend to encapsulate both the dealer and the various different kinda of players into this one class.


In [13]:
# Dear beloved random library, I welcome you to yet another project of mine.
import random


class Player:

    # The constructor accepts the behavior, the number of cards to start off with
    # since it's 2 for the player and 1 for the dealer. It also accepts a boolean
    # for whether to treat the ace as 1 or 11.
    def __init__(
        self,
        behavior: PlayerBehavior,
        initial_cards: int,
        treat_ace_as_1: bool,
    ):
        # The constructor immediately stores these values.
        self.behavior = behavior
        self.treat_ace_as_1 = treat_ace_as_1
        self.standing = False

        # This part is interesting because I must first initiate a pool of cards
        # before I can call the hit method which accepts an integer dictating
        # how many cards to hit.
        self.cards: list[Card] = []
        self.hit(initial_cards)

    # Intended for debugging. Renders the deck as emojis.
    def deck_string(self):
        return " ".join(str(card) for card in self.cards)

    # Surprisingly, you can just use list() on an enum class to get its values.
    # And lists of course play nicely with random.choice(). I am starting to
    # like Python but God, do I miss curly braces. Also note that this is
    # a static method. There is no reason for it to be bound to an instance
    # as that's the job of the hit method. This just pick a random card. Now,
    # if I did the extra credit, this would be a part of an instance of
    # the BlackJack class but sorry, I don't have time for that.
    def draw():
        return random.choice(list(Card))

    # This is actually a method intended for opponents to use. So a player looking
    # at the upcard of the dealer or players looking at each other's cards.
    def upcard(self):
        return self.cards[0]

    # Uses the static draw method and appends the card.
    def hit(self, hits=1):
        for _ in range(hits):
            self.cards.append(Player.draw())

    # Just swap the boolean.
    def stand(self):
        self.standing = True

    # Sums the worths of the cards to get the worth of the player. Really happy with
    # how this method turned out.
    def worth(self):
        return sum(card.worth(self.treat_ace_as_1) for card in self.cards)

    # The worth method makes this one so simple to implement.
    def is_bust(self):
        return self.worth() > 21

    # This is where the player decides to hit or stand. The player also has access
    # to the instance of the opponent. Technically speaking, the player can
    # access opponent opponent.cards to view the whole deck and I am unsure how
    # you would prevent that since Python does not have private methods but
    # I say let's have faith in humanity and trust that the player only uses
    # opponent.upcard() at most. Plus I designed the method so I am complying
    # with the rules.
    def decide(self, opponent: "Player"):
        if self.behavior == PlayerBehavior.DRUNK_GAMBLER:

            # The drunk player does a 50/50 on hitting or standing. The easiest
            # way to achieve this is by simply generating a random number from
            # 0 to 1 and seeing if it's below or above 0.5.
            if random.random() < 0.5:
                self.hit()
            else:
                self.stand()

        # Like I talked about before, this guy hits unit it's a 50/50 chance that
        # hitting once more will result in a bust.
        elif self.behavior == PlayerBehavior.HIGH_SCHOOL_STATISTICIAN:

            # This is awful for performance since I am computing the average over
            # and over again but oh well, this is never going to be deployed onto
            # a server or anything.
            values = [card.worth(self.treat_ace_as_1) for card in Card]
            average = sum(values) / len(values)

            # 21 less the average less 1 should be right before busting.
            if self.worth() < 21 - average - 1:
                self.hit()
            else:
                self.stand()

        elif self.behavior == PlayerBehavior.SEASONED_RICH_GUY:
            # The basic strategy involves first checking out the upcard of the
            # dealer. My upcard method makes it easy and then I just get its
            # worth.
            dealer_card = opponent.upcard().worth()
            total = self.worth()

            # The strategy depends on how we treat aces.
            if self.treat_ace_as_1:
                # These thresholds just seems to be heuristics that players have
                # built over the years. Not much to say here than I am just following
                # a blog haha.
                if total <= 11:
                    self.hit()
                elif 12 <= total <= 16 and dealer_card >= 7:
                    self.hit()
                else:
                    self.stand()
            else:
                if total <= 17:
                    self.hit()
                # Using the same in array trick here to check for specific values of
                # the dealer's upcard worth.
                elif total == 18 and dealer_card in [9, 10, 11]:
                    self.hit()
                else:
                    self.stand()

        # The dealer of course just hits over ang over again until 17 is surpassed.
        elif self.behavior == PlayerBehavior.DEALER:
            if self.worth() < 17:
                self.hit()
            else:
                self.stand()

    # This is just the player loop that the main game logic will use. So I go looping
    # until the player is standing or is busted. Of course checking this is made easy
    # due to the boolean and the function.
    def play(self, opponent: "Player"):
        while not self.standing and not self.is_bust():
            self.decide(opponent)

Now that I am done making the `Player` class, it is time to consider what I will need for the game class. One thing that comes to mind is dictating the end state of the game. So that's why I have made the `GameResult` class.


In [14]:
class GameResult(Enum):
    # I have already done enums so idk what to say here.
    PLAYER_WINNER = 0
    DEALER_WINNER = 1
    DRAW = 2

    # This is where I assign how much money you get for winning, losing, or drawing.
    # For now, I am just doing simple, -1, 0, and 1 points.
    def worth(self):

        # Switch cases for the win!
        case = {
            GameResult.PLAYER_WINNER: 1,
            GameResult.DEALER_WINNER: -1,
            GameResult.DRAW: 0,
        }

        return case[self]

Time to set up the game logic! There is surprisingly not a lot of code here since by the rules of the assignment, the player does first and then the dealer, not one after the another for every card.


In [15]:
class BlackJack:
    # The behavior here is noted for the player. The dealer just gets the hard coded
    # dealer behavior. Furthermore, the treat_ace_as_1 is an argument I also noted here.
    def __init__(
        self, player_behavior=PlayerBehavior.DRUNK_GAMBLER, treat_ace_as_1=True
    ):
        # Though the player_behavior here is passed, making it data driven, the 2 starting
        # cards is hard coded.
        self.player = Player(
            player_behavior,
            2,
            treat_ace_as_1,
        )

        # Of course the dealer is completely hard coded (of course the treat_ace_as_1
        # passed as is).
        self.dealer = Player(
            PlayerBehavior.DEALER,
            1,
            treat_ace_as_1,
        )

    # The play method has no loop directly here though there's a while loop in the
    # player class.
    def play(self):

        # So we let the player play first and pass the dealer's instance as the
        # opponent.
        self.player.play(self.dealer)

        # If they're a bust, we just give the dealer a win.
        if self.player.is_bust():
            return GameResult.DEALER_WINNER

        # And since I used a return, I don't need an else here. Nevertheless, now
        # that we know the player is not a bust, we let the dealer play.
        self.dealer.play(self.player)

        # There's two ways the player can win is if the dealer busts or if the player
        # has more worth than the dealer.
        if self.dealer.is_bust() or self.player.worth() > self.dealer.worth():
            return GameResult.PLAYER_WINNER

        # The player loses if the dealer has more worth than the player.
        if self.player.worth() < self.dealer.worth():
            return GameResult.DEALER_WINNER

        # And, once again, since all other branches of logic have returned, I don't
        # need an if or else here; I just return a draw.
        return GameResult.DRAW

Now that the game logic has been implemented, you can go about debugging it with those emojis from a while ago. The assignment wants a CSV file showing the results of the game. But the assignment wants a class that handles this? Okay, let's do that.


In [16]:
class GameMatrix:
    # Take a note of the number of rounds we are going to play with each variation.
    # I could hard code it to 100 as the assignment calls for it but it's better
    # to be dynamic.
    def __init__(self, rounds: int):
        self.rounds = rounds

    def csv(self):
        # The first row of the csv is the header. The first column is the round number.
        draft = "Round, "

        # Then I add the variations of player behavior and treat_ace_as_1.
        draft += ", ".join(
            f"{player_behavior} ({"Ace=1" if treat_ace_as_1 else "Ace=11"})"
            for player_behavior in PlayerBehavior
            for treat_ace_as_1 in [True, False]
        )

        # New line.
        draft += "\n"

        # Looping through the number of rounds.
        for i in range(0, self.rounds):
            index = i + 1

            # First column is the round number here.
            draft += f"{index}, "

            # Player behavior loop.
            for player_behavior in PlayerBehavior:

                # The treat_ace_as_1 loop is also here.
                for treat_ace_as_1 in [True, False]:

                    # Play the game in this current variation.
                    game = BlackJack(player_behavior, treat_ace_as_1)
                    result = game.play()

                    # Add the result to the draft.
                    draft += f"{result.worth()}, "

            # New line again here.
            draft += "\n"

        # Return the draft which should be a valid csv.
        return draft

Time to use the function!


In [ ]:
# The content of the csv for 100 rounds.
text = GameMatrix(100).csv()

# Print it for debugging.
print(text)

# And, finally, write it to a file.
with open("test.games.csv", "w") as f:
    f.write(text)
    f.close()

Round, Drunk Gambler (Ace=1), Drunk Gambler (Ace=11), High School Statistician (Ace=1), High School Statistician (Ace=11), Seasoned Rich Guy (Ace=1), Seasoned Rich Guy (Ace=11), Dealer (Ace=1), Dealer (Ace=11)
1, -1, -1, -1, 1, 0, -1, -1, 1, 
2, -1, 1, 1, 1, -1, 1, -1, 0, 
3, -1, -1, -1, 1, 0, -1, -1, -1, 
4, 1, -1, -1, 1, 1, 1, -1, 1, 
5, 1, -1, 1, 1, -1, 1, 1, 1, 
6, -1, 1, -1, -1, 1, -1, 1, -1, 
7, 1, -1, 1, -1, 1, -1, -1, -1, 
8, -1, -1, -1, 1, 1, -1, 0, -1, 
9, -1, -1, -1, -1, 1, -1, 1, 1, 
10, -1, -1, -1, 1, -1, -1, 1, -1, 
11, -1, -1, 1, 0, -1, -1, -1, -1, 
12, -1, 1, -1, 1, 1, 1, 0, -1, 
13, -1, -1, 1, 1, 1, -1, 1, -1, 
14, -1, -1, -1, -1, -1, -1, -1, 1, 
15, -1, -1, -1, 0, -1, -1, -1, 1, 
16, -1, -1, -1, 1, -1, -1, -1, 1, 
17, -1, 1, -1, -1, -1, -1, 1, 1, 
18, -1, -1, 1, 1, -1, 1, 0, 0, 
19, -1, 1, -1, 1, 1, 0, -1, -1, 
20, 1, -1, 0, -1, -1, 0, 0, 1, 
21, -1, 1, -1, 1, 1, -1, 1, 1, 
22, -1, -1, 1, -1, -1, 0, -1, -1, 
23, -1, -1, -1, 1, -1, 1, 1, -1, 
24, -1, -1, -1, -1, -1, 1,

To save you some time, here's a run I did on Friday uploaded to Google Sheets: https://docs.google.com/spreadsheets/d/1QLnzs7N4hXCWY2EpTaZq5YAZbh5USvadVlXYtIN-6OQ
